In [3]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("house-price-experiment")

with mlflow.start_run():
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_metric("rmse", 24027.12)
    print("Run logged!")

2026/04/30 22:05:16 INFO mlflow.tracking.fluent: Experiment with name 'house-price-experiment' does not exist. Creating a new experiment.


Run logged!
🏃 View run stylish-fly-1000 at: http://127.0.0.1:5000/#/experiments/2/runs/4f736245fac14f09b2cc7a4bbb4a014f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_csv("data/train.csv")

In [5]:
def clean_data(df):
    df = df.copy()
    
    none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                 "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                 "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", 
                 "BsmtFinType2", "MasVnrType"]
    for col in none_cols:
        df[col] = df[col].fillna("None")
    
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
    df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
    df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
    df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])
    
    return df

df_clean = clean_data(df)

In [6]:
quality_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ordinal_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
                "HeatingQC", "KitchenQual", "FireplaceQu",
                "GarageQual", "GarageCond", "PoolQC"]
for col in ordinal_cols:
    df_clean[col] = df_clean[col].map(quality_map)

df_encoded = pd.get_dummies(df_clean, drop_first=True)

In [7]:
X = df_encoded.drop(columns=["SalePrice", "Id"])
y = df_encoded["SalePrice"]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data ready. Train size:", X_train.shape)

Data ready. Train size: (1168, 229)


In [8]:
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("house-price-experiment")

models = {
    "LinearRegression": (LinearRegression(), {}),
    "XGBoost": (XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=42), 
                {"n_estimators": 1000, "learning_rate": 0.05}),
    "LightGBM": (LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42),
                 {"n_estimators": 1000, "learning_rate": 0.05}),
}

for model_name, (model, params) in models.items():
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        
        mlflow.log_param("model", model_name)
        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)
        
        print(f"{model_name} RMSE: {round(rmse, 2)}")

LinearRegression RMSE: 52720.12
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/2/runs/c6836cfdb71448a082040e518eb60b74
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
XGBoost RMSE: 25278.73
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/2b580b5ce6284b02a767d2a7e154cd55
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002604 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3160
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 150
[LightGBM] [Info] Start training from score 181441.541952
LightGBM RMSE: 29952.54
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/2/runs/aca91e3b306a4905bf18d580480904e7
🧪 V

In [9]:
xgb_best = XGBRegressor(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=1000,
    subsample=0.8,
    random_state=42
)

with mlflow.start_run(run_name="XGBoost_Tuned"):
    xgb_best.fit(X_train, y_train)
    preds = xgb_best.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    
    mlflow.log_params({
        "learning_rate": 0.05,
        "max_depth": 3,
        "n_estimators": 1000,
        "subsample": 0.8
    })
    mlflow.log_metric("rmse", rmse)
    
    print(f"Tuned XGBoost RMSE: {round(rmse, 2)}")

Tuned XGBoost RMSE: 24027.12
🏃 View run XGBoost_Tuned at: http://127.0.0.1:5000/#/experiments/2/runs/9f9bce3f25294fe6a79f7573dbc973c8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [10]:
with mlflow.start_run(run_name="XGBoost_Tuned_v2"):
    xgb_best.fit(X_train, y_train)
    preds = xgb_best.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    
    mlflow.log_params({
        "learning_rate": 0.05,
        "max_depth": 3,
        "n_estimators": 1000,
        "subsample": 0.8
    })
    mlflow.log_metric("rmse", rmse)
    mlflow.xgboost.log_model(xgb_best, "model")
    
    print(f"RMSE: {round(rmse, 2)}")
    print("Model saved to MLflow")

2026/04/30 22:23:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/30 22:23:11 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/nikhilvaishnav/Documents/Update portfolio/house-price-ml
2026/04/30 22:23:11 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/nikhilvaishnav/Documents/Update portfolio/house-price-ml
2026/04/30 22:23:11 INFO mlflow.utils.environment: Detected uv project at /Users/nikhilvaishnav/Documents/Update portfolio/house-price-ml. Attempting to export requirements via 'uv export'.
2026/04/30 22:23:11 INFO mlflow.utils.uv_utils: Exported 205 dependencies via uv
2026/04/30 22:23:11 INFO mlflow.utils.environment: Successfully exported 205 requirements from uv project. Skipping package capture based inference.
2026/04/30 22:23:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml envi

RMSE: 24027.12
Model saved to MLflow
🏃 View run XGBoost_Tuned_v2 at: http://127.0.0.1:5000/#/experiments/2/runs/699111de15e94dfbbbdd9ba9d457f3bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
